# Arm 4: Full Feature Bank, Nested Selection, XGBoost/RF

**Objective.** Test whether the full feature bank (5015 columns: band power, PLI, coherence, PLV, Kuramoto) with nested feature selection captures predictive signal beyond FAA alone (Arm 1), and whether nonlinear classifiers (XGBoost, RF) outperform the linear/shrinkage models given that flexibility - per Decision 5's pre-specified evaluation set.

**Inputs.**
- `full_cohort_features.parquet` (160 x 5031, already assembled and QC'd)
- Age, responder status (`cohort_filtered_n163.xlsx`, unchanged from Arm 1)

**Method.** Mutual-information univariate filter (`SelectKBest`, `mutual_info_classif`), k tuned jointly with each classifier's own hyperparameters inside the same `GridSearchCV` call (`select__k` alongside `clf__C` etc.), via an `sklearn.Pipeline`. k-grid: [10, 25, 50, 100, 200, 300, 500]. All six classifiers (LDA, elastic-net, L2, unregularized logistic, XGBoost, RF) route through this search uniformly - including LDA/unregularized, which previously had no `param_grid` at all in `run_nested_cv`; now every classifier searches at minimum over k.

**Assumptions.**
- New function (`run_nested_cv_with_selection`) built and validated here, not added to `run_nested_cv` in `src/modelling.py` directly - avoids changing validated, production behavior for Arms 1/2/5/6/Bailey. Whether this moves to `modelling.py` afterward depends on future reuse, not decided yet.
- Mutual information chosen over ANOVA F specifically because this arm's purpose includes testing nonlinear signal (XGBoost/RF); an F-test filter could discard nonlinear-only features before the tree models ever see them.
- `mutual_info_classif`'s internal randomness is seeded explicitly (`random_state=RANDOM_STATE`), since it has its own stochastic component independent of the outer CV/GridSearchCV seeding already in place.
- Compute cost is expected to be substantially higher than any prior arm (5015-column MI scoring per fold, six classifiers, k as an added grid dimension) - anticipated to run long, consistent with this project's documented pattern for full-cohort runs.

In [1]:
# Imports
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from functools import partial

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing import find_repo_root
from src.modelling import AgeDeconfounder, run_nested_cv
import time
from joblib import Memory
import tempfile

project_root = find_repo_root()
data_dir = project_root / "data"
features_dir = data_dir / "features"

RANDOM_STATE = 42

# Load the full assembled feature bank and the cohort's age/Responder labels
full_features_df = pd.read_parquet(features_dir / "full_cohort_features.parquet")
cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")

print(full_features_df.shape)  # expect (160, 5031)

(160, 5031)


In [2]:
# defining feature_cols
id_qc_cols = [
    'subject_id', 'condition', 'variant', 'reason', 'heog_variant',
    'preprocessing_status', 'n_epochs_before', 'n_epochs_after',
    'output_path', 'autoreject_consensus', 'autoreject_n_interpolate',
    'autoreject_extreme', 'heog_n_candidates', 'heog_n_valid',
    'heog_correction_applied', 'preprocessing_error',
]

feature_cols = [c for c in full_features_df.columns if c not in id_qc_cols]
print(len(feature_cols))  # expect 5015

5015


In [3]:
# Build model_df: full feature bank + age + Responder, same merge pattern as every other arm
model_df = full_features_df[['subject_id'] + feature_cols].merge(
    cohort_df[['TDBRAIN_ID', 'age', 'Responder']],
    left_on='subject_id', right_on='TDBRAIN_ID', how='left'
).drop(columns='TDBRAIN_ID')

print(len(model_df))                                          # expect 160
print(model_df[['age', 'Responder']].isna().sum())             # expect all 0
print(model_df[feature_cols].isna().sum().sum())                # expect 0 - no NaNs across 5015 feature columns
print(model_df['Responder'].value_counts())                     # expect ~93/67, matching every other arm

X_arm4 = model_df[feature_cols].values
y_arm4 = model_df['Responder'].values
age_arm4 = model_df['age'].values

print(X_arm4.shape)  # expect (160, 5015)

160
age          0
Responder    0
dtype: int64
0
Responder
1    93
0    67
Name: count, dtype: int64
(160, 5015)


In [4]:
# Wraps a classifier + its param grid into a (selector, classifier) Pipeline,
# so k is tuned jointly with the classifier's own hyperparameters rather than
# fixed outside the search - every classifier, including LDA/unregularized
# (previously param_grid=None), now searches at minimum over k.
#
# Pipeline(memory=...) caches each step's fitted output keyed by its inputs
# and parameters - so when only the downstream classifier's hyperparameters
# change (not select__k), the cached MI-selection result is reused instead of
# recomputed. Without this, mutual_info_classif over 5015 columns gets
# recomputed on every single grid combination, even ones that share the same
# k - the dominant cost for classifiers with grids beyond just k (elastic-net,
# L2, XGBoost, RF).a

cache_dir = tempfile.mkdtemp()
memory = Memory(location=cache_dir, verbose=0)

K_GRID = [10, 25, 50, 100, 200, 300, 500]
mi_scorer = partial(mutual_info_classif, random_state=RANDOM_STATE)

def build_pipeline_and_grid(estimator, param_grid, k_grid=K_GRID):
    pipeline = Pipeline([
        ('select', SelectKBest(score_func=mi_scorer)),
        ('clf', estimator),
    ], memory=memory)
    combined_grid = {'select__k': k_grid}
    if param_grid is not None:
        combined_grid.update({f'clf__{key}': val for key, val in param_grid.items()})
    return pipeline, combined_grid


scale_pos_weight = (y_arm4 == 0).sum() / (y_arm4 == 1).sum()  # ~0.72, fixed cohort-level ratio

classifier_specs_arm4 = {
    'LDA (Ledoit-Wolf, balanced priors)': build_pipeline_and_grid(
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto', priors=[0.5, 0.5]),
        None
    ),
    'Logistic (unregularized, balanced)': build_pipeline_and_grid(
        LogisticRegression(C=np.inf, max_iter=1000, class_weight='balanced'),
        None
    ),
    'Logistic (elastic-net, balanced)': build_pipeline_and_grid(
        LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.5, 0.9]}
    ),
    'Logistic (L2 / "Bayesian" MAP, balanced)': build_pipeline_and_grid(
        LogisticRegression(l1_ratio=0, max_iter=1000, class_weight='balanced'),
        {'C': [0.01, 0.1, 1, 10, 100]}
    ),
    'XGBoost (balanced via scale_pos_weight)': build_pipeline_and_grid(
        XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, eval_metric='logloss'),
        {'n_estimators': [100, 300], 'max_depth': [2, 4], 'learning_rate': [0.01, 0.1]}
    ),
    'Random Forest (balanced)': build_pipeline_and_grid(
        RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
        {'n_estimators': [100, 300], 'max_depth': [3, 5, None], 'min_samples_leaf': [1, 5, 10]}
    ),
}

for name, (pipeline, grid) in classifier_specs_arm4.items():
    n_combos = np.prod([len(v) for v in grid.values()])
    print(f"{name}: {n_combos} combinations")

LDA (Ledoit-Wolf, balanced priors): 7 combinations
Logistic (unregularized, balanced): 7 combinations
Logistic (elastic-net, balanced): 105 combinations
Logistic (L2 / "Bayesian" MAP, balanced): 35 combinations
XGBoost (balanced via scale_pos_weight): 56 combinations
Random Forest (balanced): 126 combinations


In [5]:
def run_nested_cv_with_selection(X, y, age, classifier_specs, n_outer_splits, n_inner_splits, random_state):
    """
    Same structure as run_nested_cv (src/modelling.py): fold-scoped age
    deconfounding, fold-scoped scaling, then fit/tune each classifier and
    score on the held-out test fold.

    Differs in one respect: classifier_specs values are (pipeline, grid)
    tuples, not (estimator, param_grid) - every classifier is wrapped in a
    Pipeline (SelectKBest + classifier) by build_pipeline_and_grid, so k is
    tuned jointly with the classifier's own hyperparameters. Every classifier
    now goes through GridSearchCV unconditionally (even LDA/unregularized,
    which had param_grid=None in run_nested_cv) - there is no longer a
    "no tuning needed" case, since select__k always needs searching.

    Parameters: same as run_nested_cv, except classifier_specs values are
    (pipeline, grid) tuples from build_pipeline_and_grid.

    Returns:
    pd.DataFrame, one row per (classifier, outer fold): classifier, fold,
    balanced_accuracy, accuracy, auc, sensitivity, specificity, ppv,
    best_params (the fold's selected k and hyperparameters - kept for
    checking selection stability across folds, unlike run_nested_cv).
    """
    outer_cv = StratifiedKFold(n_splits=n_outer_splits, shuffle=True, random_state=random_state)
    results = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        age_train, age_test = age[train_idx], age[test_idx]

        deconf = AgeDeconfounder()
        deconf.fit(X_train, age_train)
        X_train_clean = deconf.transform(X_train, age_train)
        X_test_clean = deconf.transform(X_test, age_test)

        scaler = StandardScaler()
        X_train_clean = scaler.fit_transform(X_train_clean)
        X_test_clean = scaler.transform(X_test_clean)

        for clf_name, (pipeline, grid) in classifier_specs.items():
            inner_cv = StratifiedKFold(n_splits=n_inner_splits, shuffle=True, random_state=random_state)
            search = GridSearchCV(pipeline, grid, cv=inner_cv, scoring='balanced_accuracy')
            search.fit(X_train_clean, y_train)
            fitted_model = search.best_estimator_

            y_pred = fitted_model.predict(X_test_clean)
            y_proba = fitted_model.predict_proba(X_test_clean)[:, 1]

            tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
            ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan

            results.append({
                'classifier': clf_name,
                'fold': fold_idx,
                'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
                'accuracy': accuracy_score(y_test, y_pred),
                'auc': roc_auc_score(y_test, y_proba),
                'sensitivity': sensitivity,
                'specificity': specificity,
                'ppv': ppv,
                'best_params': search.best_params_,
            })

    return pd.DataFrame(results)

In [6]:
# Time test: single fold, elastic-net (75 combinations: 5 select__k x 5 C x
# 3 l1_ratio) - the classifier where Pipeline(memory=...) caching should show
# a real difference, since select__k repeats across every C/l1_ratio pair,
# unlike LDA where each k only appears once.

N_OUTER_SPLITS = 5
N_INNER_SPLITS = 3

outer_cv_time_test = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_cv_time_test.split(X_arm4, y_arm4))

X_train, X_test = X_arm4[train_idx], X_arm4[test_idx]
y_train, y_test = y_arm4[train_idx], y_arm4[test_idx]
age_train, age_test = age_arm4[train_idx], age_arm4[test_idx]

deconf = AgeDeconfounder()
deconf.fit(X_train, age_train)
X_train_clean = deconf.transform(X_train, age_train)
X_test_clean = deconf.transform(X_test, age_test)

scaler = StandardScaler()
X_train_clean = scaler.fit_transform(X_train_clean)
X_test_clean = scaler.transform(X_test_clean)

pipeline, grid = classifier_specs_arm4['Logistic (elastic-net, balanced)']
inner_cv = StratifiedKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

start = time.time()
search = GridSearchCV(pipeline, grid, cv=inner_cv, scoring='balanced_accuracy')
search.fit(X_train_clean, y_train)
elapsed = time.time() - start

fitted_model = search.best_estimator_
y_pred = fitted_model.predict(X_test_clean)
y_proba = fitted_model.predict_proba(X_test_clean)[:, 1]

print(f"Elapsed: {elapsed:.1f}s for elastic-net, 1 fold, 75 combinations x 3 inner folds")
print(f"best_params: {search.best_params_}")
print(f"balanced_accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"predictions: {y_pred}")
print(f"probabilities: {np.round(y_proba, 3)}")
print(f"y_test:       {y_test}")
print(f"\nRough per-combination time: {elapsed/75:.2f}s (elastic-net has 75 combos)")

Elapsed: 100.2s for elastic-net, 1 fold, 75 combinations x 3 inner folds
best_params: {'clf__C': 0.1, 'clf__l1_ratio': 0.9, 'select__k': 300}
balanced_accuracy: 0.6111
predictions: [0 1 0 0 1 1 1 0 1 0 1 1 1 0 1 0 0 1 0 1 1 1 1 1 1 1 1 0 0 1 0 1]
probabilities: [0.312 0.587 0.488 0.448 0.562 0.559 0.518 0.191 0.543 0.462 0.532 0.558
 0.653 0.477 0.633 0.248 0.45  0.546 0.462 0.682 0.564 0.674 0.64  0.55
 0.547 0.598 0.597 0.374 0.379 0.581 0.456 0.526]
y_test:       [0 0 1 1 0 1 0 0 1 1 0 1 1 1 0 0 0 1 0 1 0 1 1 1 1 1 0 0 0 1 1 1]

Rough per-combination time: 1.34s (elastic-net has 75 combos)


In [7]:
# All-six-classifier check, single outer fold - before committing to the full
# 5-fold run_nested_cv_with_selection call. Reuses the same fold split as the
# elastic-net timing test, looping over every classifier in classifier_specs_arm4
# rather than calling run_nested_cv_with_selection directly, since that function
# runs all 5 outer folds internally and would commit to the full runtime before
# this check could catch a problem.

outer_cv_check = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_cv_check.split(X_arm4, y_arm4))

X_train, X_test = X_arm4[train_idx], X_arm4[test_idx]
y_train, y_test = y_arm4[train_idx], y_arm4[test_idx]
age_train, age_test = age_arm4[train_idx], age_arm4[test_idx]

deconf = AgeDeconfounder()
deconf.fit(X_train, age_train)
X_train_clean = deconf.transform(X_train, age_train)
X_test_clean = deconf.transform(X_test, age_test)

scaler = StandardScaler()
X_train_clean = scaler.fit_transform(X_train_clean)
X_test_clean = scaler.transform(X_test_clean)

inner_cv = StratifiedKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_start = time.time()
for clf_name, (pipeline, grid) in classifier_specs_arm4.items():
    clf_start = time.time()
    search = GridSearchCV(pipeline, grid, cv=inner_cv, scoring='balanced_accuracy')
    search.fit(X_train_clean, y_train)
    clf_elapsed = time.time() - clf_start

    fitted_model = search.best_estimator_
    y_pred = fitted_model.predict(X_test_clean)

    n_combos = np.prod([len(v) for v in grid.values()])
    print(f"{clf_name}: {clf_elapsed:.1f}s ({n_combos} combos), "
          f"best_params={search.best_params_}, "
          f"balanced_accuracy={balanced_accuracy_score(y_test, y_pred):.4f}")

fold_elapsed = time.time() - fold_start
print(f"\nTotal for one outer fold, all 6 classifiers: {fold_elapsed:.1f}s")
print(f"Rough full-run estimate (x5 outer folds, observed-data pass only): {fold_elapsed*5/60:.1f} min")

LDA (Ledoit-Wolf, balanced priors): 3.6s (7 combos), best_params={'select__k': 50}, balanced_accuracy=0.6667
Logistic (unregularized, balanced): 3.4s (7 combos), best_params={'select__k': 200}, balanced_accuracy=0.5397
Logistic (elastic-net, balanced): 33.2s (105 combos), best_params={'clf__C': 0.1, 'clf__l1_ratio': 0.9, 'select__k': 300}, balanced_accuracy=0.6111
Logistic (L2 / "Bayesian" MAP, balanced): 1.1s (35 combos), best_params={'clf__C': 100, 'select__k': 200}, balanced_accuracy=0.5675
XGBoost (balanced via scale_pos_weight): 24.2s (56 combos), best_params={'clf__learning_rate': 0.1, 'clf__max_depth': 4, 'clf__n_estimators': 300, 'select__k': 200}, balanced_accuracy=0.5317
Random Forest (balanced): 46.3s (126 combos), best_params={'clf__max_depth': 3, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 300, 'select__k': 50}, balanced_accuracy=0.6310

Total for one outer fold, all 6 classifiers: 111.7s
Rough full-run estimate (x5 outer folds, observed-data pass only): 9.3 min


In [8]:
# Arm 4 observed-data pass: full feature bank, nested MI selection, all six
# classifiers, all 5 outer folds. This is the ~9.3 min run estimated from the
# single-fold check - needed before the permutation test, since observed_arm4
# (inside the permutation cell) is computed from arm4_results_df's actual
# balanced accuracy, not assumed.

arm4_results_df = run_nested_cv_with_selection(
    X_arm4, y_arm4, age_arm4, classifier_specs_arm4,
    N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE
)

print(arm4_results_df.groupby('classifier')[['balanced_accuracy', 'accuracy', 'auc', 'sensitivity', 'specificity', 'ppv']].mean())
arm4_results_df

                                          balanced_accuracy  accuracy  \
classifier                                                              
LDA (Ledoit-Wolf, balanced priors)                 0.511018   0.51875   
Logistic (L2 / "Bayesian" MAP, balanced)           0.498284   0.50625   
Logistic (elastic-net, balanced)                   0.529947   0.53750   
Logistic (unregularized, balanced)                 0.445103   0.44375   
Random Forest (balanced)                           0.552574   0.56875   
XGBoost (balanced via scale_pos_weight)            0.521621   0.53750   

                                               auc  sensitivity  specificity  \
classifier                                                                     
LDA (Ledoit-Wolf, balanced priors)        0.538082     0.573684     0.448352   
Logistic (L2 / "Bayesian" MAP, balanced)  0.519340     0.561404     0.435165   
Logistic (elastic-net, balanced)          0.553155     0.581871     0.478022   
Logistic (unreg

,classifier,fold,balanced_accuracy,accuracy,auc,sensitivity,specificity,ppv,best_params
0,"LDA (Ledoit-Wolf, balanced priors)",0,0.666667,0.68750,0.793651,0.833333,0.500000,0.681818,{'select__k': 50}
1,"Logistic (unregularized, balanced)",0,0.539683,0.56250,0.615079,0.722222,0.357143,0.590909,{'select__k': 200}
2,"Logistic (elastic-net, balanced)",0,0.611111,0.62500,0.690476,0.722222,0.500000,0.650000,"{'clf__C': 0.1, 'clf__l1_ratio': 0.9, 'select_..."
3,"Logistic (L2 / ""Bayesian"" MAP, balanced)",0,0.567460,0.59375,0.615079,0.777778,0.357143,0.608696,"{'clf__C': 100, 'select__k': 200}"
4,XGBoost (balanced via scale_pos_weight),0,0.531746,0.56250,0.575397,0.777778,0.285714,0.583333,"{'clf__learning_rate': 0.1, 'clf__max_depth': ..."
5,Random Forest (balanced),0,0.630952,0.65625,0.638889,0.833333,0.428571,0.652174,"{'clf__max_depth': 3, 'clf__min_samples_leaf':..."
6,"LDA (Ledoit-Wolf, balanced priors)",1,0.511905,0.53125,0.500000,0.666667,0.357143,0.571429,{'select__k': 100}
7,"Logistic (unregularized, balanced)",1,0.519841,0.53125,0.490079,0.611111,0.428571,0.578947,{'select__k': 100}
8,"Logistic (elastic-net, balanced)",1,0.492063,0.50000,0.480159,0.555556,0.428571,0.555556,"{'clf__C': 1, 'clf__l1_ratio': 0.5, 'select__k..."
9,"Logistic (L2 / ""Bayesian"" MAP, balanced)",1,0.456349,0.46875,0.507937,0.555556,0.357143,0.526316,"{'clf__C': 0.1, 'select__k': 100}"


In [9]:
# Permutation test: Arm 4, checkpointed given the ~15hr expected runtime.
# Saves null_scores_arm4 to disk every CHECKPOINT_EVERY permutations, so a
# kernel crash, laptop sleep, or interruption loses at most that many
# permutations' worth of progress, not the whole run. Resumable: if a saved
# checkpoint exists, loads it and continues from where it left off rather
# than restarting at 0.

import pickle
from pathlib import Path

N_PERMUTATIONS = 100
CHECKPOINT_EVERY = 5
checkpoint_path = features_dir / "arm4_permutation_checkpoint.pkl"

rng = np.random.RandomState(RANDOM_STATE)

observed_arm4 = arm4_results_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()

if checkpoint_path.exists():
    with open(checkpoint_path, 'rb') as f:
        checkpoint = pickle.load(f)
    null_scores_arm4 = checkpoint['null_scores_arm4']
    start_perm = checkpoint['completed_permutations']
    print(f"Resuming from checkpoint: {start_perm}/{N_PERMUTATIONS} permutations already done")
else:
    null_scores_arm4 = {name: [] for name in classifier_specs_arm4}
    start_perm = 0
    print(f"Starting fresh: N_PERMUTATIONS = {N_PERMUTATIONS}")

for i in range(start_perm, N_PERMUTATIONS):
    y_shuffled = rng.permutation(y_arm4)
    perm_df = run_nested_cv_with_selection(X_arm4, y_shuffled, age_arm4, classifier_specs_arm4,
                                            N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_result = perm_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()
    for name, score in perm_result.items():
        null_scores_arm4[name].append(score)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == N_PERMUTATIONS:
        with open(checkpoint_path, 'wb') as f:
            pickle.dump({'null_scores_arm4': null_scores_arm4, 'completed_permutations': i + 1}, f)
        print(f"Checkpoint saved: {i+1}/{N_PERMUTATIONS} permutations complete")

print(f"\n{'classifier':<45} {'observed':>10} {'null mean':>10} {'p-value':>10}")
for name in classifier_specs_arm4:
    null_arr = np.array(null_scores_arm4[name])
    p_value = (np.sum(null_arr >= observed_arm4[name]) + 1) / (N_PERMUTATIONS + 1)
    print(f"{name:<45} {observed_arm4[name]:>10.4f} {null_arr.mean():>10.4f} {p_value:>10.4f}")

Starting fresh: N_PERMUTATIONS = 100


/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/

Checkpoint saved: 5/100 permutations complete


/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/

Checkpoint saved: 10/100 permutations complete


...

Checkpoint saved: 90/100 permutations complete


/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/

Checkpoint saved: 95/100 permutations complete


/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/miniconda3/envs/eeg-rtms/

Checkpoint saved: 100/100 permutations complete

classifier                                      observed  null mean    p-value
LDA (Ledoit-Wolf, balanced priors)                0.5110     0.4983     0.3663
Logistic (unregularized, balanced)                0.4451     0.5023     0.9307
Logistic (elastic-net, balanced)                  0.5299     0.5003     0.2178
Logistic (L2 / "Bayesian" MAP, balanced)          0.4983     0.5020     0.5347
XGBoost (balanced via scale_pos_weight)           0.5216     0.5042     0.3366
Random Forest (balanced)                          0.5526     0.5117     0.1386


## Arm 4 (Full Feature Bank, Nested Selection, XGBoost/RF) - Findings

**Observed pass (5 outer folds, mutual-information filter selection tuned jointly with each classifier's hyperparameters, k grid [10,25,50,100,200,300,500]):** mean balanced accuracy 0.445-0.553 across the six classifiers, with severe fold-to-fold instability - e.g. LDA ranges 0.312-0.667 across the 5 outer folds, unregularized logistic 0.285-0.540. Selected `k` also varies substantially fold to fold with no consistent pattern (LDA: 50, 100, 25, 25, 200 across the 5 folds). This is consistent with, not contrary to, the instability at this p:n ratio (n≈128 per training fold, 5015 candidate features) already anticipated in Decision 5, citing Chang et al. (2025) and Varoquaux (2018).

Permutation test (N=100, not 1,000 - a documented reduction from every other arm's standard, due to compute cost at this feature-bank scale; see Residual assumptions), same nested-selection procedure rerun per shuffle:

| Classifier | Observed BA | Null mean | p-value |
|---|---|---|---|
| LDA (Ledoit-Wolf, balanced priors) | 0.511 | 0.498 | 0.366 |
| Logistic (unregularized, balanced) | 0.445 | 0.502 | 0.931 |
| Logistic (elastic-net, balanced) | 0.530 | 0.500 | 0.218 |
| Logistic (L2, balanced) | 0.498 | 0.502 | 0.535 |
| XGBoost (balanced via scale_pos_weight) | 0.522 | 0.504 | 0.337 |
| Random Forest (balanced) | 0.553 | 0.512 | 0.139 |

No classifier reaches significance at α=0.05. Null means cluster tightly around 0.50 across all six, confirming the permutation procedure is correctly calibrated - the null result reflects the observed data, not a broken test. Random Forest is the closest to significance (p=0.139) but this does not constitute a trend worth reading into further.

**Diagnostic control confirmed as designed.** Unregularized logistic - included specifically to illustrate degradation at high feature-to-sample ratios (Decision 5) - has both the worst observed balanced accuracy (0.445, below chance) and the highest p-value (0.931) of all six classifiers. This is the predicted outcome, not an anomaly.

**The full feature bank, with nested MI selection and nonlinear-capable classifiers (XGBoost, RF), does not recover predictive signal beyond what FAA alone (Arm 1) or FAA+Kuramoto (Arm 2) already showed.** More features and more model flexibility did not help at this sample size - a substantive negative result, not a pipeline failure, and consistent with what the small-n/high-p instability literature already predicted before this arm was run.

`saga`-based classifiers (elastic-net) hit `ConvergenceWarning` (`max_iter` reached) during multiple permutation iterations. `max_iter=5000` was already generous relative to every other arm; elastic-net's own p-value (0.218) is not a clear outlier among the six, so this is unlikely to be driving the null result, but is noted as an observed limitation rather than omitted.

**Residual assumptions**

100 permutations, not 1,000, due to the ~9.3 min/full-pass runtime at this feature-bank scale (~15.5 hours for 100 vs. an infeasible ~93 hours for 1,000). The permutation p-value floor at N=100 is ~0.0099 rather than ~0.001 - sufficient to detect significance at the conventional 0.05 threshold, but with less resolution near very small p-values than every other arm in this project. Given none of the six p-values approach 0.05, this resolution limit did not affect the substantive conclusion here.

`select__k`'s grid was extended once, from [10,25,50,100,200] to include 300 and 500, after the narrower grid showed four of six classifiers selecting the ceiling value (200) in a single-fold check - confirmed as a genuine constraint, not a coincidence, since elastic-net's selection moved to k=300 once available. No classifier selected k=500 in the extended grid's single-fold check, suggesting the search now has adequate headroom on both ends.

Feature-selection stability itself (which specific features get selected, not just k) was not examined - only k's value and downstream accuracy were tracked. A full instability analysis was out of scope for this arm.